In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.5 MB/s eta 0:00:00


In [ ]:
import torch
import pickle
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random
import re
import torch
from collections import Counter

**Change to false for similarity based examples:**

In [ ]:
use_cluster_examples = False

**Change to false for test set calculation:**

In [ ]:
use_validation = False

## LOAD JUDGE MODEL

In [ ]:
model_id = "AtlaAI/Selene-1-Mini-Llama-3.1-8B"

In [ ]:
# Configuration for 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # A100/L4 GPU support bfloat16
)

In [ ]:
# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

In [ ]:
# Load Model with Flash Attention (makes model faster and handle more text without running OOM)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    #attn_implementation="flash_attention_2",
    dtype=torch.bfloat16
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
tokenizer.model_max_length = 200000
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.eos_token_id

Load the correct files:

In [ ]:
if use_validation:
  with open("INSTANCES_100_VAL.pkl", "rb") as f:
    data = pickle.load(f)
  if use_cluster_examples:
    with open("EXAMPLES_300_VAL_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
    with open("LLAMA_300_VAL_CLUSTER.pkl", "rb") as f:
      llama_sums = pickle.load(f)
  else:
    with open("EXAMPLES_300_VAL_SIM.pkl", "rb") as f:
      examples = pickle.load(f)
    with open("LLAMA_300_VAL_SIM.pkl", "rb") as f:
      llama_sums = pickle.load(f)
else:
  with open("INSTANCES_200_TEST.pkl", "rb") as f:
    data = pickle.load(f)
  if use_cluster_examples:
    with open("EXAMPLES_600_TEST_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
    with open("LLAMA_600_Test_CLUSTER.pkl", "rb") as f:
      llama_sums = pickle.load(f)
  else:
    with open("EXAMPLES_600_TEST_SIM.pkl", "rb") as f:
      examples = pickle.load(f)
    with open("LLAMA_600_Test_SIM.pkl", "rb") as f:
      llama_sums = pickle.load(f)

In [ ]:
# Pre-group examples by instance ID (3 examples per ID)
# dictionary with keys = instance ID and value = list filled with 3 dictionaries of examples belonging to instance ID

examples_map = {}
for ex in examples:
    iid = ex['inst_id']
    if iid not in examples_map:
        examples_map[iid] = []
    examples_map[iid].append(ex)

In [ ]:
# Create a lookup dictionary: {doc_id: llama_summary_text}
llama_lookup = {item['doc_id']: item['llama_sum'] for item in llama_sums}

# Add the Llama summary to examples_map
for iid in examples_map:
    for ex in examples_map[iid]:
        did = ex['doc_id']
        # Add the 'llama_sum' to the example dictionary
        ex['llama_sum'] = llama_lookup.get(did, "No summary available")

# PHASE 1 - BASELINE ACCURACY

Define Functions:

In [ ]:
# NO EXPLANATIONS
def format_demonstrations_no_expl(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
      demo_str += f"Non-Target Summary (Avoid this style!): {ex['llama_sum']}\n\n"
    return demo_str.strip()

In [ ]:
# Find A, B, C, D, or E in the response
def extract_choice(response):
    #  Strict Format "**Result:** A"
    match = re.search(r"\*\*Result:\*\*\s*([A-E])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # Loose Format "Result: A"
    match = re.search(r"Result:\s*([A-E])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # if no match A-E
    return "FAIL"

Define Prompt:

In [ ]:
prompt_phase1 = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.
You will be provided with 3 examples of the Target Model's behavior.
For every example you will also be provided with a summary that was NOT produced by the Target Model and does NOT match the Target Model's summarization style.
Your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.
Avoid picking the candidate that was NOT produced by the Target Model.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern. Do not forget how the summary should NOT look like.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

## EVALUATION

In [ ]:
results_ph1 = {}
score_counter_ph1 = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_no_expl(current_examples)

    final_prompt = prompt_phase1.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph1[predicted_model] += 1

    print(data_idx)
    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph1[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

0
**Reasoning:** The Target Model's style is characterized by concise summaries that directly state the main points without elaboration. It often uses simple language and focuses on the key elements of the story. In the examples provided, the summaries are short and to the point, conveying the essential information without unnecessary details. To match this style, the selected summary should be similarly concise and directly state the main points without elaboration.

Step 2: Identify which candidate matches that inferred pattern. The summaries should not include extraneous information or overly complex language. Candidate E stands out as it directly states the main points without elaboration, matching the Target Model's style.

**Result:** E 

Predicted: E (llama) 

1
**Reasoning:** The Target Model consistently produces summaries that are concise and directly state the key information. They often lack elaborate descriptions or contextual details. In Example 1, the summary is brief an

Display Results:

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph1['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph1.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph1['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph1['FAIL']}")


JUDGE REPORT
Accuracy: 9.00% 

Distribution:
  Llama: 72
  Gold: 16
  Distilbart: 9
  Flan: 3


# PHASE 2 - INCLUDED EXPLANATIONS

In [ ]:
# Find A, B, C, D, or E in the response
def extract_choice(response):
    #  Strict Format "**Result:** A"
    match = re.search(r"\*\*Result:\*\*\s*([A-E])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # Loose Format "Result: A"
    match = re.search(r"Result:\s*([A-E])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # if no match A-E
    return "FAIL"

# EXPLANATION - TOP 2 SENTENCES

In [ ]:
# TOP TWO SENTENCES
def format_demonstrations_top2(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      top_sentences = ex.get("top_2_sentences", [])
      formatted_influential = "\n".join([f"- {s['sentence'].strip()}" for s in top_sentences])
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Most Influential Sentences: {formatted_influential}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
      demo_str += f"Non-Target Summary (Avoid this style!): {ex['llama_sum']}\n\n"
    return demo_str.strip()

## EVALUATION

In [ ]:
prompt_sent_top2 = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.
You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Sentences: The specific sentences from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.
4. A Non-Target Summary hat was NOT produced by the Target Model and does NOT match the Target Model's summarization style.
For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.
Avoid picking the candidate that was NOT produced by the Target Model.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern. Do not forget how the summary should NOT look like.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

In [ ]:
results_ph2_sent_top2 = {}
score_counter_ph2_sent_top2 = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_top2(current_examples)

    final_prompt = prompt_sent_top2.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph2_sent_top2[predicted_model] += 1


    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph2_sent_top2[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
"As a consequence, this could be contributing to the rising incidence of type-1 diabetes, which is appearing in ever younger age groups.
"Adapt will use a medication to protect the beta cells from the stress, so that they survive longer."
The researchers say if it is successful, the trial will offer a cost-effective way of preventing type-1 diabetes that could be made available immediately to children at risk.
The trial has initial funding from the type-1 diabetes charity JDRF.
Scotland has the third highest rate of type-1 diabetes in the world, and a good system of record to identify affected families.
The study will start recruitment in Tayside and will then extend across Scotland before crossing into England.
There are two main types of diabetes:
Type-1 diabetes can develop at any age, but usually appears before the age of 40, particularly in childhood.
About 10% of all diabetes is type-1, but it is the most common t

KeyboardInterrupt: 

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph2_sent_top2['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph2_sent_top2.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph2_sent_top2['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph2_sent_top2['FAIL']}")


JUDGE REPORT
Accuracy: 8.00% 

Distribution:
  Llama: 77
  Gold: 9
  Distilbart: 8
  Flan: 6


# EXPLANATION - TOP SENTENCE

In [ ]:
# TOP SENTENCE
def format_demonstrations_top(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Most Influential Sentence: {ex['top_sentence']['sentence']}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
      demo_str += f"Non-Target Summary (Avoid this style!): {ex['llama_sum']}\n\n"
    return demo_str.strip()

In [ ]:
prompt_sent_top = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.
You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Sentence: The specific sentence from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.
4. A Non-Target Summary hat was NOT produced by the Target Model and does NOT match the Target Model's summarization style.
Your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.
Avoid picking the candidate that was NOT produced by the Target Model.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.  Do not forget how the summary should NOT look like.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

## EVALUATION

In [ ]:
results_ph2_sent = {}
score_counter_ph2_sent = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_top(current_examples)

    final_prompt = prompt_sent_top.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph2_sent[predicted_model] += 1

    print(data_idx)
    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph2_sent[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

0
**Reasoning:** The Target Model tends to focus on key details and phrases that summarize the main points of the article. It often uses a concise style, highlighting the most critical information. To match this style, the summary should be brief and directly address the central issue. 

Step 1: Identify the common style or pattern from the three examples. The summaries provided by the Target Model are concise and focus on the main charges or actions. They avoid unnecessary details and emphasize the key points. 

Step 2: Evaluate which candidate matches the inferred pattern. Candidate D directly addresses the main issue of the article, mentioning the charges and the scheduled trial, while also providing context about the charity and the individuals involved. This aligns with the concise style observed in the Target Model's summaries.

Step 3: Avoid the non-target style by ensuring the summary does not include extraneous information or overly elaborate language. Candidate D effectively 

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph2_sent['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph2_sent.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph2_sent['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph2_sent['FAIL']}")


JUDGE REPORT
Accuracy: 10.00% 

Distribution:
  Llama: 74
  Gold: 11
  Distilbart: 10
  Flan: 5


# EXPLANATION - TOP PHRASE (per refined sentence)

In [ ]:
def format_demonstrations_phrase(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
        input = ex["doc"]
        top_phrases = ex.get("top_phrases", {})
        sorted_phrases = sorted(
            top_phrases.values(),
            key=lambda x: x["score"],
            reverse=True
        )
        formatted_influential = ", ".join( item["phrase"].strip() for item in sorted_phrases)
        demo_str += f"Example {idx+1}:\n"
        demo_str += f"Input: {input}\n\n"
        demo_str += f"Most Influential Phrases: {formatted_influential}\n\n"
        demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
        demo_str += f"Non-Target Summary (Avoid this style!): {ex['llama_sum']}\n\n"


    return demo_str.strip()

In [ ]:
prompt_phrase = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.
You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Phrases: The specific phrases from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.
4. A Non-Target Summary hat was NOT produced by the Target Model and does NOT match the Target Model's summarization style.
Your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.
Avoid picking the candidate that was NOT produced by the Target Model.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.  Do not forget how the summary should NOT look like.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

## EVALUATION

In [ ]:
results_ph2_phrase = {}
score_counter_ph2_phrase = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_phrase(current_examples)

    final_prompt = prompt_phrase.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph2_phrase[predicted_model] += 1

    print(data_idx)
    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph2_phrase[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

0
**Reasoning:** The Target Model consistently focuses on key phrases that summarize the main points of the input articles. It tends to highlight specific details that are crucial to understanding the context, such as names, charges, and dates. To match this style, the selected summary should emphasize these elements while maintaining brevity. 

Step 1: Identify the common style or pattern from the three examples. The Target Model emphasizes key phrases, such as "men who have pleaded guilty," "Mr Whyte's lawyer," and "the infamous £25m jewellery heist." It also tends to keep the summaries concise.

Step 2: Evaluate the candidate summaries against the inferred pattern. Response B stands out because it directly addresses the main points of the input article, including the names of the individuals involved, the charges, and the trial date. It maintains a concise style similar to the Target Model's summaries.

**Result:** B 

Predicted: B (llama) 

1
**Reasoning:** The Target Model consist

KeyboardInterrupt: 

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph2_phrase['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph2_phrase.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph2_phrase['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph2_phrase['FAIL']}")


# EXPLANATION - TOP 2 PHRASES (per refined sentence)

In [ ]:
def format_demonstrations_phrase_top2(example_list):
    demo_str = ""

    for idx, ex in enumerate(example_list):
        input = ex["doc"]

        top_2_dict = ex["top_2_phrases"]

        all_phrases = [
            item["phrase"].strip()
            for phrase_list in top_2_dict.values()
            for item in phrase_list
        ]

        formatted_influential = ", ".join(all_phrases)
        demo_str += f"Example {idx+1}:\n"
        demo_str += f"Input: {input}\n\n"
        demo_str += f"Most Influential Phrases: {formatted_influential}\n\n"
        demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
        demo_str += f"Non-Target Summary (Avoid this style!): {ex['llama_sum']}\n\n"

    return demo_str.strip()

In [ ]:
prompt_phrase_top2 = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.
You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Phrases: The specific phrases from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.
4. A Non-Target Summary hat was NOT produced by the Target Model and does NOT match the Target Model's summarization style.
Your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.
Avoid picking the candidate that was NOT produced by the Target Model.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.  Do not forget how the summary should NOT look like.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

In [ ]:
results_ph2_phrase_top2 = {}
score_counter_ph2_phrase_top2 = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_phrase_top2(current_examples)

    final_prompt = prompt_phrase_top2.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph2_phrase_top2[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph2_phrase_top2[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.

Example 2:
Input: Four men, aged 17, 18, 25 and 27, were injured during the incident, which took place in Whatriggs Road in the early hours of Saturday.
They were taken to the town's Crosshouse Hospital for treatment and were later released.
Police said four men, aged 17, 18, 19 and 20, had been arrested in connection with the incident.
They are expected to appear at Kilmarnock Sheriff Court on Monday.

Most Influential Phrases: in the early hours of Saturday, in Whatriggs Road, to appear at Kilmarnock Sheriff Court on Monday, are

Target Summary:  Four men have been arrested in connection with an assault in Kilmarnock.

Non-Target Summary (Avoid this style!): Four men, aged 17 to 27, were injured in an incident in Whatriggs Road, Kilmarnock, and four others, aged 17 to 20, were arrested and expected to appear in court.

Example 3:
Input: The 28-year-old male officer suffered serious burns and is in an induced coma, pr

KeyboardInterrupt: 

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph2_phrase_top2['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph2_phrase_top2.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph2_phrase_top2['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph2_phrase_top2['FAIL']}")


JUDGE REPORT
Accuracy: 7.00% 

Distribution:
  Llama: 77
  Gold: 11
  Distilbart: 7
  Flan: 5
